# prism RL Environment — GRPO Training Demo
## Meta × PyTorch × Hugging Face OpenEnv Hackathon 2026
Training Qwen2.5-3B on the prism multi-agent reliability environment.

In [ ]:
!pip install openenv-core trl transformers torch wandb rich requests -q

In [ ]:
ENV_URL = "https://gauthamram-prism.hf.space"
import requests
health = requests.get(f"{ENV_URL}/health").json()
print(f"Environment status: {health}")

In [ ]:
import requests, json

res = requests.post(f"{ENV_URL}/reset", json={
    "seed": 42,
    "options": {"task_domain": "debug", "agents": 2, "failure_rate": 0.0}
})
obs = res.json()
episode_id = obs["episode_id"]
print(f"Episode started: {episode_id[:8]}...")
print(f"Task graph: {list(obs['task_graph'].keys())}")

SEQUENCE = [
    {"tool": "checkpoint",   "args": {}},
    {"tool": "research_web", "args": {"q": "analyse Python bug"}},
    {"tool": "write_code",   "args": {"path": "fix.py", "body": "def calculate_sum(items):\n    return sum(items)"}},
    {"tool": "critique",     "args": {"target": "verified fix"}},
    {"tool": "finish",       "args": {"answer": "range(len(items)) fix applied"}},
]

rewards = []
for action in SEQUENCE:
    r = requests.post(f"{ENV_URL}/step", json={"action": action, "episode_id": episode_id})
    data = r.json()
    reward = data["reward"]
    rewards.append(reward)
    bd = data["info"]["reward_breakdown"]
    print(f"  {action['tool']:15} reward={reward:.4f}  "
          f"progress={bd['progress_delta']:.2f}  "
          f"atomic={bd['atomic_health']:.2f}  "
          f"terminal={bd['terminal_bonus']:.2f}")

print(f"\nTotal episode reward: {sum(rewards):.4f}")

In [ ]:
import json, time, os
import matplotlib.pyplot as plt

N_EPISODES = 30
all_rewards = []
rolling_rewards = []
rolling = 0.3

for ep in range(1, N_EPISODES + 1):
    res = requests.post(f"{ENV_URL}/reset", json={
        "seed": 100 + ep,
        "options": {"task_domain": "debug", "agents": 2, "failure_rate": 0.0}
    })
    eid = res.json()["episode_id"]

    ep_total = 0.0
    for action in SEQUENCE:
        r = requests.post(f"{ENV_URL}/step",
                          json={"action": action, "episode_id": eid})
        ep_total += r.json().get("reward", 0.0)

    rolling = 0.1 * ep_total + 0.9 * rolling
    all_rewards.append(ep_total)
    rolling_rewards.append(rolling)
    print(f"[{ep:3d}/{N_EPISODES}] ep_reward={ep_total:.4f}  rolling={rolling:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_rewards, alpha=0.4, color='blue', label='Episode Reward')
ax1.plot(rolling_rewards, color='blue', linewidth=2, label='Rolling Average')
ax1.axhline(y=max(all_rewards), color='green', linestyle='--',
            label=f'Best: {max(all_rewards):.4f}')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Episode Reward')
ax1.set_title('prism RL Training — Reward Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Transfer scores from /metrics
metrics = requests.get(f"{ENV_URL}/metrics").json()
transfer = metrics.get("transfer_scores", [])
if transfer:
    domains = list(set(t["domain"] for t in transfer))
    colors = {"debug": "green", "market_research": "blue", "etl": "orange"}
    for d in domains:
        pts = [(t["eval_episode"], t["score"]) for t in transfer if t["domain"] == d]
        if pts:
            xs, ys = zip(*pts)
            ax2.plot(xs, ys, label=d, color=colors.get(d, "gray"), marker='o')
ax2.set_xlabel('Episode')
ax2.set_ylabel('Transfer Score')
ax2.set_title('Cross-Domain Generalization')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('prism_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as prism_training_curves.png")

In [ ]:
GROQ_KEY   = "YOUR_GROQ_KEY"
GOOGLE_KEY = "YOUR_GOOGLE_KEY"

models = [
    {"provider": "groq",   "model": "llama-3.3-70b-versatile", "key": GROQ_KEY},
    {"provider": "gemini", "model": "gemini-2.0-flash",         "key": GOOGLE_KEY},
]

results = {}
for cfg in models:
    res = requests.post(f"{ENV_URL}/reset", json={
        "seed": 42,
        "options": {"task_domain": "debug", "agents": 2, "failure_rate": 0.0}
    })
    eid = res.json()["episode_id"]

    requests.post(f"{ENV_URL}/models/config", json={
        "provider": cfg["provider"], "model": cfg["model"],
        "api_key": cfg["key"], "episode_id": eid
    })

    total = 0.0
    for _ in range(15):
        r = requests.post(f"{ENV_URL}/step",
                          json={"action": {"tool": "checkpoint", "args": {}},
                                "episode_id": eid})
        d = r.json()
        total += d.get("reward", 0.0)
        if d.get("terminated"):
            break

    results[cfg["model"]] = total
    print(f"{cfg['model']}: {total:.4f}")

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(results.keys(), results.values(),
       color=["#4CAF50", "#2196F3"], alpha=0.8)
ax.set_ylabel("Cumulative Reward")
ax.set_title("Model Agnosticism Proof — Same Environment, Different LLMs")
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()